# Reranker negatives (Phase 1e) — L40 + vLLM

Generates deliberately bad quatrains for training the trait-fit classifier in the reranker, four kinds for every training trait:

| Type | What it is |
|---|---|
| `off_topic` | a fine rhyming quatrain that is not a roast and not about the person |
| `flat` | about the trait, but flat and unfunny |
| `non_rhyming` | a roast about the trait whose lines do not rhyme |
| `wrong_trait` | a normal roast of a different trait, labelled with this one |

`prepare_negatives.py` later adds the disses dropped in the manual review (1b). About 1,400 quatrains.

**Upload next to this notebook:** `traits.json` from `data/` in the repo.

**Workflow:** `MODE = "pilot"` → Run All → read the samples → `MODE = "full"` → Restart & Run All → copy `llm_out/negatives.jsonl` and `llm_out/negatives_meta.json` to `data/llm_out/` in the repo → after `prepare_disses.py finalize`, run `python data/prepare_negatives.py`.

**Model:** Qwen/Qwen3-32B-FP8 (Apache-2.0).

## 1. Install

In [ ]:
%pip install -q vllm

## 2. Settings

In [ ]:
import os, re, json, math, time, random, itertools
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"   # skip FlashInfer's sampler, which needs nvcc >= 12.8 to compile
from collections import Counter, defaultdict
from datetime import date

# ======================= THE SWITCH =======================
MODE = "pilot"              # "pilot" = a few requests to read the output | "full" = the real run
# ==========================================================
assert MODE in ("pilot", "full")

MODEL_ID = "Qwen/Qwen3-32B-FP8"           # same model as the quatrain corpus; thinking switched off below
CHAT_KW  = {"chat_template_kwargs": {"enable_thinking": False}}
SEED, TEMPERATURE, TOP_P = 42, 0.9, 0.95
MAX_TOKENS = 2048

NEG_PER_REQUEST = 10           # quatrains per request: one request per training trait and per type

OUT_DIR = "llm_out" if MODE == "full" else "llm_pilot_out"
os.makedirs(OUT_DIR, exist_ok=True)
print("MODE:", MODE, "| output:", OUT_DIR)

## 3. Traits

In [ ]:
TRAITS  = json.load(open("traits.json", encoding="utf-8"))
TRAIN   = [t for t, v in TRAITS.items() if not v["heldout"]]
HELDOUT = [t for t, v in TRAITS.items() if v["heldout"]]
print(f"{len(TRAIN)} training traits | held out: {HELDOUT}")

## 4. Load the model

If it runs out of memory, lower `gpu_memory_utilization` to 0.85.

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL_ID, max_model_len=4096, gpu_memory_utilization=0.90,
          enable_prefix_caching=True, seed=SEED)
params = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS, repetition_penalty=1.05)

def chat(msgs):
    # one reply per conversation -> (texts, output tokens)
    if not msgs:
        return [], 0
    outs = llm.chat(msgs, params, use_tqdm=True, **CHAT_KW)
    return [o.outputs[0].text for o in outs], sum(len(o.outputs[0].token_ids) for o in outs)

## 5. Prompts

In [ ]:
QUOTE_MAP = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"'})

def normalize(text):
    return text.translate(QUOTE_MAP)        # curly quotes -> straight, as in form_checks.py

THINK_RE = re.compile(r"<think>.*?</think>", re.S)

def parse_json_array(text):
    m = re.search(r"\[.*\]", THINK_RE.sub("", text), re.S)
    try:
        v = json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None
    return v if isinstance(v, list) else None

def dump(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)

TAG_RE = re.compile(r"^\[([^\]]*)\]\s*")

def parse_lines(item):
    # an object with "lines", or a bare list of 4 strings; strips the leading [end word] tag
    lines = item.get("lines") if isinstance(item, dict) else item
    if not isinstance(lines, list):
        return []
    return [TAG_RE.sub("", normalize(str(l)).strip()).strip() for l in lines if str(l).strip()]

PERSONAS = ["sarcastic stand-up comedian", "rapper in a roast battle", "disappointed teacher",
            "brutally honest best friend", "over-the-top game show host"]

def describe(t):
    return f'"{t}" (also said as: {", ".join(TRAITS[t]["synonyms"][:2])}; flip side: {TRAITS[t]["flip"]})'

NEG_DESC = {
    "off_topic":   "rhyming AABB quatrains that are not roasts at all and have nothing to do with the person; "
                   "they are about the weather, food, traffic or similar",
    "flat":        "AABB quatrains about the trait that are flat and unfunny: they state plain facts or mild "
                   "compliments, with no joke and no flip side",
    "non_rhyming": "four-line roasts about the trait where the lines do not rhyme at all",
}

def neg_prompt(trait, kind, k):
    return (f"Write {k} {NEG_DESC[kind]}. The trait is {describe(trait)}.\n"
            "These are bad examples for training a quality filter, so make them bad in exactly this way "
            "and otherwise normal: four short lines, modern English, no swear words, nothing hateful about "
            "identity.\nReturn a JSON array of arrays, each holding the 4 lines as strings.")

def wrong_trait_prompt(trait, k, persona):
    # a normal roast of `trait`; the run labels it with a different trait
    return (f"Write {k} different roasts of a person who says they are {describe(trait)}. "
            "Turn the positive trait into its flip side.\n"
            f'Voice: a {persona}, speaking directly to the person ("you").\n'
            "Each roast is an AABB quatrain: line 1 rhymes with line 2, line 3 rhymes with line 4. "
            "Short, punchy lines. No swear words, nothing hateful about identity.\n"
            "Return a JSON array of arrays, each holding the 4 lines as strings.")

print(neg_prompt("confident", "flat", NEG_PER_REQUEST))

## 6. Run

In [ ]:
rng, jobs, t0 = random.Random(SEED + 3), [], time.time()
for t in (TRAIN[:2] if MODE == "pilot" else TRAIN):
    for kind in NEG_DESC:
        jobs.append((kind, t, None, [{"role": "user", "content": neg_prompt(t, kind, NEG_PER_REQUEST)}]))
    other = rng.choice([x for x in TRAIN if x != t])
    jobs.append(("wrong_trait", t, other,
                 [{"role": "user", "content": wrong_trait_prompt(other, NEG_PER_REQUEST, rng.choice(PERSONAS))}]))
print(f"{len(jobs)} requests")
texts, tokens = chat([m for *_, m in jobs])
neg_rows, bad = [], 0
for (kind, t, other, _), text in zip(jobs, texts):
    items = parse_json_array(text)
    if items is None:
        bad += 1
        continue
    for it in items:
        lines = parse_lines(it)
        if lines:
            neg_rows.append({"type": kind, "traits": [t], "about": other, "lines": lines})
dump(f"{OUT_DIR}/negatives.jsonl", neg_rows)
stats = {"quatrains": len(neg_rows), "by_type": dict(Counter(r["type"] for r in neg_rows)),
         "bad_json": bad, "tokens": tokens, "seconds": round(time.time() - t0, 1)}
print(f"{len(neg_rows):,} negatives | {stats['by_type']} | bad JSON {bad}")

## 7. Samples

Each one should be bad in exactly its own way and otherwise normal.

In [ ]:
for r in random.Random().sample(neg_rows, min(8, len(neg_rows))):
    print(f"{r['type']} | {r['traits']}" + (f" (actually about {r['about']})" if r["about"] else ""))
    print("   " + "\n   ".join(r["lines"]) + "\n")

## 8. Save the run record

In [ ]:
meta = {"date": date.today().isoformat(), "mode": MODE, "model": MODEL_ID, "license": "Apache-2.0",
        "temperature": TEMPERATURE, "top_p": TOP_P, "seed": SEED,
        "settings": {"NEG_PER_REQUEST": NEG_PER_REQUEST, "types": list(NEG_DESC) + ["wrong_trait"], "PERSONAS": PERSONAS},
        "prompts": {"flat_example": neg_prompt("confident", "flat", NEG_PER_REQUEST),
                    "wrong_trait_example": wrong_trait_prompt("funny", NEG_PER_REQUEST, PERSONAS[0])},
        "stats": stats}
json.dump(meta, open(f"{OUT_DIR}/negatives_meta.json", "w", encoding="utf-8"), indent=2)
print("Saved to", OUT_DIR)